# Answer Filter + COT Fixer

1. **Fix COT format** — remove `The answer in \boxed{–} is` noise
2. **Check 1** — `answer` column matches train.csv ground truth
3. **Check 2** — final `\boxed{}` in `generated_cot` matches ground truth
4. Save filtered CSV (only rows where both checks pass)

In [57]:
# ── CONFIG ────────────────────────────────────────────────────────────────
INPUT_CSV         = "dataset_generated.csv"          # CSV to fix + filter
TRAIN_CSV         = "F:\\Hackathons\\Kaggle-Nemotron\\data_manipulation\\dataset_generated.csv"           # ground-truth
OUTPUT_FIXED_CSV  = "dataset_fixed.csv"              # COT-fixed (all rows)
OUTPUT_FILTERED_CSV = "new_dataset_filtered2.csv" # COT-fixed + correct only

# Column names in INPUT_CSV
INPUT_ANSWER_COL  = "answer"       # answer column to check (Check 1)
INPUT_COT_COL     = "generated_cot" # COT column to fix + extract boxed from (Check 2)

# Join key: 'id' or 'prompt' (must exist in both CSVs)
JOIN_ON           = "id"

# FILTER_MODE:
#   'both'   - keep rows where answer col AND boxed BOTH match GT
#   'answer' - keep rows where answer col matches GT only
#   'boxed'  - keep rows where boxed matches GT only
#   'either' - keep rows where at least one matches
FILTER_MODE       = "both"

# Normalise before comparing (strip, lowercase, collapse whitespace)
NORMALIZE         = True

# Absolute numeric tolerance: match if abs(pred - gt) <= NUMERIC_TOL
# Set None to disable
NUMERIC_TOL       = 1e-6

# Relative numeric tolerance: match if abs(pred - gt) / abs(gt) <= NUMERIC_REL_TOL
# e.g. 0.01 = within 1% of GT — catches rounded boxed answers like 154.631 vs 154.62
# Set None to disable
NUMERIC_REL_TOL   = 0.01
# ──────────────────────────────────────────────────────────────────────────

In [58]:
import pandas as pd
import re
from pathlib import Path

base = Path(".")

df = pd.read_csv(base / INPUT_CSV, encoding="utf-8")
df_train = pd.read_csv(base / TRAIN_CSV, encoding="utf-8")

print(f"Input  : {len(df):,} rows  cols={df.columns.tolist()}")
print(f"Train  : {len(df_train):,} rows  cols={df_train.columns.tolist()}")

Input  : 9,500 rows  cols=['id', 'prompt', 'answer', 'type', 'generated_cot']
Train  : 9,500 rows  cols=['id', 'prompt', 'answer', 'type', 'generated_cot']


## Step 1 — Fix COT format

**Bad pattern** (what the model generates):
```
I will now return the answer in \boxed{}
The answer in \boxed{–} is \boxed{cat imagines book}
```
**Fixed**:
```
I will now return the answer in \boxed{}
The answer is \boxed{cat imagines book}
```

In [59]:
# Matches: "The answer in \boxed{<anything>} is " (the garbage prefix)
# Leaves:  "The answer is \boxed{ACTUAL_ANSWER}"
_BAD_PATTERN = re.compile(
    r'The answer in \\boxed\{[^}]*\} is (\\boxed\{[^}]*\})'
)
_GOOD_REPLACEMENT = r'The answer is \1'

def fix_cot(cot):
    if pd.isna(cot):
        return cot
    return _BAD_PATTERN.sub(_GOOD_REPLACEMENT, str(cot))

n_before = df[INPUT_COT_COL].apply(
    lambda x: bool(_BAD_PATTERN.search(str(x))) if pd.notna(x) else False
).sum()

df[INPUT_COT_COL] = df[INPUT_COT_COL].apply(fix_cot)

n_after = df[INPUT_COT_COL].apply(
    lambda x: bool(_BAD_PATTERN.search(str(x))) if pd.notna(x) else False
).sum()

print(f"Rows with bad pattern before fix : {n_before:,}")
print(f"Rows with bad pattern after  fix : {n_after:,}")

# Verify on a sample
print("\n--- Sample fixed COT tail ---")
print(repr(df[INPUT_COT_COL].iloc[0][-120:]))

Rows with bad pattern before fix : 9,500
Rows with bad pattern after  fix : 0

--- Sample fixed COT tail ---
' NOT6 = NOT(0) = 1\n6 NOT7 = NOT(0) = 1\n7 C1 = 1\n\nI will now return the answer in \\boxed{}\nThe answer is \\boxed{10010111}'


In [60]:
# Save COT-fixed version (all rows, regardless of answer correctness)
out_fixed = base / OUTPUT_FIXED_CSV
df.to_csv(out_fixed, index=False, encoding="utf-8")
print(f"Saved fixed CSV ({len(df):,} rows) → {out_fixed.resolve()}")

Saved fixed CSV (9,500 rows) → F:\Hackathons\Kaggle-Nemotron\data_manipulation\dataset_fixed.csv


## Step 2 — Build ground-truth lookup from train.csv

In [61]:
assert JOIN_ON in df_train.columns, f"'{JOIN_ON}' not in train.csv cols: {df_train.columns.tolist()}"
assert JOIN_ON in df.columns,       f"'{JOIN_ON}' not in input cols: {df.columns.tolist()}"

lookup = dict(zip(df_train[JOIN_ON], df_train["answer"]))
df["_gt"] = df[JOIN_ON].map(lookup)

unmatched = df["_gt"].isna().sum()
print(f"Lookup size    : {len(lookup):,}")
print(f"Matched to GT  : {df['_gt'].notna().sum():,}")
print(f"No GT match    : {unmatched:,}")

Lookup size    : 9,500
Matched to GT  : 9,500
No GT match    : 0


## Step 3 — Extract final `\boxed{}` from COT

In [62]:
# Brace-balanced extractor — handles nested braces like \boxed{\frac{1}{2}}
def extract_last_boxed(text):
    if pd.isna(text):
        return None
    s = str(text)
    last_start = None
    i = 0
    while i < len(s):
        idx = s.find(r'\boxed{', i)
        if idx == -1:
            break
        last_start = idx + len(r'\boxed{')
        i = last_start
    if last_start is None:
        return None
    # Walk forward counting braces
    depth, j = 1, last_start
    while j < len(s) and depth > 0:
        if s[j] == '{':
            depth += 1
        elif s[j] == '}':
            depth -= 1
        j += 1
    return s[last_start:j-1].strip() if depth == 0 else None

df["_boxed_ans"] = df[INPUT_COT_COL].apply(extract_last_boxed)

missing_boxed = df["_boxed_ans"].isna().sum()
print(f"Rows with no \\boxed{{}} in COT : {missing_boxed:,}")
print(df[[INPUT_ANSWER_COL, "_boxed_ans", "_gt"]].head(5).to_string())

Rows with no \boxed{} in COT : 79
                  answer             _boxed_ans                    _gt
0               10010111               10010111               10010111
1               01000011               00011111               01000011
2      cat imagines book      cat imagines book      cat imagines book
3                XXXVIII                XXXVIII                XXXVIII
4  wizard creates secret  wizard creates secret  wizard creates secret


## Step 4 — Compare answers

In [63]:
def normalize(s):
    if pd.isna(s):
        return ""
    return " ".join(str(s).strip().lower().split())

def try_float(s):
    try:
        return float(str(s).replace(",", ""))
    except (ValueError, TypeError):
        return None

def answers_match(pred, gt):
    if pd.isna(gt):
        return False
    a = normalize(pred) if NORMALIZE else str(pred).strip()
    b = normalize(gt)   if NORMALIZE else str(gt).strip()
    if a == b:
        return True
    if NUMERIC_TOL is not None or NUMERIC_REL_TOL is not None:
        fa, fb = try_float(a), try_float(b)
        if fa is not None and fb is not None:
            diff = abs(fa - fb)
            if NUMERIC_TOL is not None and diff <= NUMERIC_TOL:
                return True
            if NUMERIC_REL_TOL is not None and fb != 0:
                if diff / abs(fb) <= NUMERIC_REL_TOL:
                    return True
    return False

# Check 1: answer column vs GT
df["_match_answer"] = [
    answers_match(p, g)
    for p, g in zip(df[INPUT_ANSWER_COL], df["_gt"])
]

# Check 2: boxed answer from COT vs GT
df["_match_boxed"] = [
    answers_match(b, g)
    for b, g in zip(df["_boxed_ans"], df["_gt"])
]

# Consistency: do answer col and boxed agree?
df["_answer_boxed_agree"] = [
    answers_match(p, b)
    for p, b in zip(df[INPUT_ANSWER_COL], df["_boxed_ans"])
]

n_eligible = df["_gt"].notna().sum()

print(f"{'Metric':<35} {'Count':>8} {'%':>7}")
print("-" * 52)
print(f"{'Total rows':<35} {len(df):>8}")
print(f"{'Eligible (has GT)':<35} {n_eligible:>8}")
print(f"{'Check 1 — answer col correct':<35} {df['_match_answer'].sum():>8}  {df['_match_answer'].sum()/n_eligible*100:>6.1f}%")
print(f"{'Check 2 — boxed in COT correct':<35} {df['_match_boxed'].sum():>8}  {df['_match_boxed'].sum()/n_eligible*100:>6.1f}%")
print(f"{'Answer col == boxed (agree)':<35} {df['_answer_boxed_agree'].sum():>8}  {df['_answer_boxed_agree'].sum()/n_eligible*100:>6.1f}%")
print(f"\nTolerance used: abs={NUMERIC_TOL}, rel={NUMERIC_REL_TOL} ({(NUMERIC_REL_TOL or 0)*100:.1f}% of GT)")

Metric                                 Count       %
----------------------------------------------------
Total rows                              9500
Eligible (has GT)                       9500
Check 1 — answer col correct            9500   100.0%
Check 2 — boxed in COT correct          9156    96.4%
Answer col == boxed (agree)             9159    96.4%

Tolerance used: abs=1e-06, rel=0.01 (1.0% of GT)


In [64]:
# ── Discrepancy sample: answer col correct but boxed wrong (or vice versa) ─
disc = df[
    df["_gt"].notna() &
    (df["_match_answer"] != df["_match_boxed"])
]
print(f"Discrepant rows (Check1 ≠ Check2): {len(disc):,}")
if len(disc):
    display(
        disc[[INPUT_ANSWER_COL, "_boxed_ans", "_gt",
              "_match_answer", "_match_boxed"]].head(10)
    )

Discrepant rows (Check1 ≠ Check2): 344


,answer,_boxed_ans,_gt,_match_answer,_match_boxed
1,01000011,00011111,01000011,True,False
29,17/,17,17/,True,False
39,|@{,NaN,|@{,True,False
60,+},+,+},True,False
65,10010101,10111001,10010101,True,False
92,00001001,00011001,00001001,True,False
97,:}',:,:}',True,False
119,00000100,00001100,00000100,True,False
130,00111001,01111001,00111001,True,False
159,-},-,-},True,False


In [65]:
# ── Apply filter ──────────────────────────────────────────────────────────
if FILTER_MODE == "both":
    mask = df["_match_answer"] & df["_match_boxed"]
elif FILTER_MODE == "answer":
    mask = df["_match_answer"]
elif FILTER_MODE == "boxed":
    mask = df["_match_boxed"]
elif FILTER_MODE == "either":
    mask = df["_match_answer"] | df["_match_boxed"]
else:
    raise ValueError(f"Unknown FILTER_MODE: {FILTER_MODE!r}")

_drop_cols = ["_gt", "_boxed_ans", "_match_answer", "_match_boxed", "_answer_boxed_agree"]
df_filtered = (
    df[mask]
    .drop(columns=_drop_cols)
    .reset_index(drop=True)
)

out_filtered = base / OUTPUT_FILTERED_CSV
df_filtered.to_csv(out_filtered, index=False, encoding="utf-8")
print(f"Filter mode : {FILTER_MODE!r}")
print(f"Kept        : {len(df_filtered):,} / {n_eligible:,} rows ({len(df_filtered)/n_eligible*100:.1f}%)")
print(f"Saved       → {out_filtered.resolve()}")
df_filtered.head(3)

Filter mode : 'both'
Kept        : 9,156 / 9,500 rows (96.4%)
Saved       → F:\Hackathons\Kaggle-Nemotron\data_manipulation\new_dataset_filtered2.csv


,id,prompt,answer,type,generated_cot
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111,bit_manipulation,We need to deduce the transformation by matchi...
1,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book,cipher,We need to find the encryption mapping from th...
2,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII,numeral,This is an Arabic to Roman numeral conversion....


In [66]:
df = pd.read_csv("dataset_31052026.csv")
len(df)

6020

In [67]:
df.iloc[0]['generated_cot']

'We need to deduce the transformation by matching the example outputs.\nI will put my final answer inside \\boxed{}.\n\nOutput 0: 11011101\n0 1\n1 1\n2 0\n3 1\n4 1\n5 1\n6 0\n7 1\n\nOutput 1: 01101101\n0 0\n1 1\n2 1\n3 0\n4 1\n5 1\n6 0\n7 1\n\nOutput 2: 01010101\n0 0\n1 1\n2 0\n3 1\n4 0\n5 1\n6 0\n7 1\n\nOutput 3: 10000001\n0 1\n1 0\n2 0\n3 0\n4 0\n5 0\n6 0\n7 1\n\nOutput 4: 01000101\n0 0\n1 1\n2 0\n3 0\n4 0\n5 1\n6 0\n7 1\n\nOutput 5: 00001001\n0 0\n1 0\n2 0\n3 0\n4 1\n5 0\n6 0\n7 1\n\nOutput 6: 00000101\n0 0\n1 0\n2 0\n3 0\n4 0\n5 1\n6 0\n7 1\n\nOutput 7: 10110011\n0 1\n1 0\n2 1\n3 1\n4 0\n5 0\n6 1\n7 1\n\nOutput bit columns (with bitsum as hash)\n0 10010001 3\n1 11101000 4\n2 01000001 2\n3 10100001 3\n4 11000100 3\n5 11101010 5\n6 00000001 1\n7 11111111 a\n\nInput 0: 01010001\n0 0\n1 1\n2 0\n3 1\n4 0\n5 0\n6 0\n7 1\n\nInput 1: 00001001\n0 0\n1 0\n2 0\n3 0\n4 1\n5 0\n6 0\n7 1\n\nInput 2: 00010101\n0 0\n1 0\n2 0\n3 1\n4 0\n5 1\n6 0\n7 1\n\nInput 3: 11111111\n0 1\n1 1\n2 1\n3 1\n4 1\n5

In [68]:
import pandas as pd

# Load the CSV files
df1 = pd.read_csv('dataset_31052026.csv')
df2 = pd.read_csv('dataset_filtered_correct.csv')

# Stack the dataframes vertically
merged_df = pd.concat([df1, df2], ignore_index=True)

# Save the result to a new CSV file
merged_df.to_csv('merged_output.csv', index=False, encoding='utf-8-sig')

